# Saturating one spectral component, and throwing it away

`sc.modules.SaturationPrep` is the opposite of an excitation. An excitation makes transverse
magnetisation **to be read**; this makes it **to be destroyed** — nothing is rephased, a spoiler
follows immediately, and what the sequence cares about is what is left *longitudinally* when the
imaging train starts.

It is a `preparation/` module for the same reason `IRPrep` is: the folder's rule is `rf.use`, and
this pulse's is `'saturation'`.

**Output:** two `.seq` files — a spoiled GRE with fat saturation and the same GRE without it, at
one protocol, so the only difference is the preparation.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pypulseq as pp

import seqcraft as sc

opts = pp.Opts(
    max_grad=40, grad_unit='mT/m',
    max_slew=180, slew_unit='T/m/s',
    B0=2.89,
    rf_dead_time=100e-6,
    rf_ringdown_time=30e-6,
    adc_dead_time=10e-6,
)

FOV_MM, MATRIX, THICKNESS_MM = 220.0, 96, 5.0
FLIP_DEG, TR_S = 12.0, 25e-3

SEQ_DIR = Path('seq')
SEQ_DIR.mkdir(exist_ok=True)

## The chemical shift, and the one number that matters

`shift_ppm` is **signed and relative to water**. Fat is below water, so it is negative. The module
converts once — `shift_ppm × 1e-6 × B0 × γ` — and emits that as the pulse's `freq_offset`.

The reason this is worth a module rather than a line of arithmetic: a sign error here produces
legal Pulseq, legal timing, legal gradients and a perfectly ordinary-looking waveform, and
saturates **water**. Nothing downstream notices. The two published references reach the same
number by different routes — one carries the sign in the ppm constant, the other applies it at the
point of use — and mixing the conventions gets you the wrong one.

In [ ]:
fat = sc.modules.SaturationPrep(
    opts=opts,
    shift_ppm=-3.45,        # signed, relative to water
    flip_deg=110.0,         # no default: two references disagree, so it is the caller's
    duration_s=8e-3,
    bandwidth_hz=424.5,
    spoil_voxel_mm=0.1,     # 1 cycle across 0.1 mm, which is the references' spoiling
)

print(f'shift            {fat.shift_ppm:+.2f} ppm')
print(f'at B0 = {opts.B0:.2f} T    {fat.offset_hz:+.1f} Hz   (negative = below water)')
print(f'band half-width  {fat.bandwidth_hz / 2:.1f} Hz')
print(f'margin to water  {fat.band_edge_hz:.1f} Hz')
print(f'pulse centre     {fat.time_to_center() * 1e3:.2f} ms into the block')
print(f'block            {fat()}')

The margin is the number a caller is choosing when they choose a duration, whether or not they
know it. Ask for a band that reaches water and the module refuses rather than quietly saturating
the signal the scan exists to acquire.

In [ ]:
try:
    sc.modules.SaturationPrep(opts=opts, shift_ppm=-3.45, flip_deg=110.0, duration_s=8e-3,
                              bandwidth_hz=1000.0, spoil_voxel_mm=0.1)
except sc.ConfigurationError as error:
    print(str(error))

## Moving the protocol between field strengths

Because the shift is in ppm, the same protocol says the same thing at any field — the hertz
offset follows `B0` and nothing else about the pulse changes. That is the property to check when
a fat-sat protocol is ported, and it is the one a hard-coded hertz value silently breaks.

In [ ]:
print(f'{"B0 / T":>8}  {"offset / Hz":>12}  {"margin / Hz":>12}  {"centre / ms":>12}')
for b0 in (1.5, 2.89, 3.0, 7.0):
    scanner = pp.Opts(max_grad=40, grad_unit='mT/m', max_slew=180, slew_unit='T/m/s', B0=b0,
                      rf_dead_time=100e-6, rf_ringdown_time=30e-6, adc_dead_time=10e-6)
    variant = sc.modules.SaturationPrep(opts=scanner, shift_ppm=-3.45, flip_deg=110.0,
                                        duration_s=8e-3, bandwidth_hz=424.5, spoil_voxel_mm=0.1)
    print(f'{b0:8.2f}  {variant.offset_hz:12.1f}  {variant.band_edge_hz:12.1f}'
          f'  {variant.time_to_center() * 1e3:12.2f}')

At 1.5 T the margin has almost vanished — **8 Hz** of clearance, against 212 Hz at 2.89 T and
816 Hz at 7 T. The protocol is still legal there and the module still builds it, but the
separation an 8 ms pulse can achieve is barely enough: fat and water are simply closer together in
hertz at lower field. A 1.5 T protocol wants a longer pulse, and the margin is the number that
says so before anyone scans.

## In composition

The preparation is played once per repetition, before the imaging kernel. `SaturationPrep` does
not know that, and should not: **when it is played and what follows it are the caller's.**

In [ ]:
gre = sc.modules.GRE2DTR(opts=opts, fov_mm=FOV_MM, matrix=(MATRIX, MATRIX),
                         thickness_mm=THICKNESS_MM, flip_deg=FLIP_DEG,
                         bandwidth_hz_px=260.0, tr_s=None)

prep_s = float(fat().duration)


def scan(*, saturate):
    """The same GRE, with and without the preparation in front of each repetition."""
    out = sc.LogicBlock('fat_sat_gre' if saturate else 'plain_gre')
    per_tr = (prep_s if saturate else 0.0) + TR_S
    for index, line in enumerate(range(MATRIX)):
        start = index * per_tr
        if saturate:
            out.add(start, fat())
        out.add(start + (prep_s if saturate else 0.0), gre(line=line, phase_deg=117.0 * index))
    return out


trees = {name: scan(saturate=name == 'fat_sat') for name in ('fat_sat', 'plain')}
for name, tree in trees.items():
    print(f'{name:8s} {tree.duration:.2f} s')

## What the emitted file says

The check that matters is not what the API was called — it is what the `.seq` declares. A
saturation pulse emitted as an `excitation` would tell a reader, a scanner and any analysis tool
the wrong thing about the experiment.

In [ ]:
for name, tree in trees.items():
    seq = sc.compile(tree, opts, name=f'{name}_2d', definitions={
        'FOV': [FOV_MM / 1e3, FOV_MM / 1e3, THICKNESS_MM / 1e3],
    })
    seq.write(str(SEQ_DIR / f'{name}_2d.seq'))
    rf = seq.get_block(1).rf
    print(f'{name:8s} block 1 rf: use={rf.use!r:14s} freq_offset={rf.freq_offset:+9.1f} Hz'
          f'   {len(seq.block_events):5d} blocks -> seq/{name}_2d.seq')

In [ ]:
fig, ax = plt.subplots(figsize=(8.4, 3.2))
t_ms = np.asarray(fat.rf.t) * 1e3 + fat.rf.delay * 1e3
ax.plot(t_ms, np.abs(fat.rf.signal), lw=1.2, label='|B1|')
ax.axvline(fat.time_to_center() * 1e3, color='crimson', ls='--', lw=1,
           label='effective centre')
ax.axvspan(t_ms[-1], t_ms[-1] + fat.spoiler.duration * 1e3, color='0.85', label='spoiler')
ax.set(xlabel='time / ms', ylabel='|B1| / a.u.',
       title='the pulse, its centre, and the spoiler that immediately follows')
ax.legend(loc='upper right', fontsize=8)
fig.tight_layout()

## What this notebook established

| | |
|---|---|
| the shift is **signed, relative to water** | `-3.45 ppm` → `-424.5 Hz` at 2.89 T, converted once and readable off the compiled file |
| the module refuses a band that reaches water | including, correctly, the same protocol at 1.5 T |
| a protocol in ppm ports across field strengths | the offset follows `B0` and nothing else moves |
| the emitted pulse declares `use='saturation'` | checked on the compiled sequence, not on the constructor |
| no gradient, no rephaser, spoiler immediately after | the absences are the contract |
| **when it is played is the caller's** | the preparation is added by the loop, not by the module |

Deliberately **not** here: CEST saturation trains, spatial saturation slabs and water excitation.
Each would need its own evidence that it shares a physical solve with this one, and "prepare, then
spoil" is a waveform silhouette rather than a shared solve.